# Project #15 — Zero-to-Hero: Production-Grade Document Q&A System

This notebook teaches enterprise local RAG with **Ollama + ChromaDB + Hybrid Retrieval**.

## Why this corpus?
We use mixed-domain documents (technical, policy, research, finance) because enterprise assistants must answer cross-domain questions and compare documents, not just summarize one policy file.

## Architecture

User Question → Query Embedding (`qwen3-embedding:4b`) → Retriever (Vector/BM25/Hybrid) → Context Builder → Prompt Template → Generator (`qwen3.5:4b`) → Citations → Evaluation + Judge (`granite4.1:3b`)

### Why RecursiveCharacterTextSplitter over fixed-size only?
Recursive splitting preserves semantic boundaries better in heterogeneous docs while still enforcing chunk budgets.

In [ ]:
from local_rag.app import load_settings
from local_rag.visualization import save_pipeline_architecture_diagram

settings = load_settings()
settings.ensure_directories()
save_pipeline_architecture_diagram(settings.diagrams_dir / "pipeline_architecture.png")
print("Saved architecture diagram")

## Step 1 — Bootstrap Realistic Corpus

We bootstrap a mixed corpus from public documentation and reports.

### Why ChromaDB over FAISS?
ChromaDB gives persistence, metadata filtering, and collection management with minimal ops overhead for beginners.

In [ ]:
# !uv run python -m local_rag bootstrap
# !uv run python -m local_rag corpus-report --profile full

## Step 2 — Ingestion + Incremental Indexing

We compute file hashes, version IDs, chunk records, embeddings, and persistent Chroma/BM25 indexes.

### Why Qwen embeddings over MiniLM?
Project constraint requires local Ollama-only embeddings and avoids Hugging Face embedding models.

In [ ]:
# !uv run python -m local_rag ingest --chunk-size 768 --chunk-overlap 100

## Step 3 — Chunking Experiments

Grid:
- chunk size: 256, 512, 768, 1024, 1500
- overlap: 0, 50, 100, 200

We measure retrieval metrics, latency, and index size to select robust defaults.

In [ ]:
# !uv run python -m local_rag run-experiments

## Step 4 — Retrieval Strategy Comparison

Compare:
- Vector retrieval
- Keyword BM25 retrieval
- Hybrid RRF retrieval

Evaluate Top-1/3/5/10 with Precision@K, Recall@K, MRR, NDCG.

In [ ]:
# !uv run python -m local_rag evaluate --strategy hybrid

## Step 5 — Prompt Engineering

Templates included:
- strict_grounding
- citation_focus
- enterprise_qa
- legal_qa
- technical_qa
- unknown_safe

### Why local inference over cloud APIs?
Local runtime protects enterprise document privacy and avoids external data transfer.

In [ ]:
# !uv run python -m local_rag query "Which document discusses encryption?" --strategy hybrid --prompt-template legal_qa --top-k 5

## Step 6 — LLM-as-a-Judge + Failure Analysis

Judge rubric:
- correctness
- groundedness
- faithfulness
- completeness
- conciseness
- citation quality

Failure demos include poor chunking, weak prompt selection, missing context, wrong retrieval strategy.

In [ ]:
# !uv run python -m local_rag failures
# !uv run python -m local_rag benchmark

## Step 7 — Run Streamlit App

```bash
uv run streamlit run streamlit_app/app.py
```

Use tabs for document management, indexing, Q&A, and analytics.

## Project Progression

- **Project #13**: educational RAG basics
- **Project #14**: local production-style RAG
- **Project #15**: enterprise document intelligence with multi-document lifecycle, hybrid retrieval, citation evidence, judge scoring, and benchmarking